In [ ]:
%load_ext autoreload
%autoreload 2

Only the GFA dataset contains a land cover attribute. This notebook 1) processes the MODIS data into GeoTiff files, 2) evaluates how well the MODIS Land Cover product can be used to predict the given categories in the USA, and 3) use the MODIS product to assign land cover categories to fires in other datasets.

### Convert MODIS data into GeoTIFF files

In [ ]:
import pandas as pd

from wildfire_stats import config as cfg
from wildfire_stats.data import classification as dcls
from wildfire_stats.data import io as dio
from wildfire_stats.data import rasters as drast
from wildfire_stats.plotting import landcover as plc
from wildfire_stats.reporting import tables as rtab

P = cfg.load_paths()
A = cfg.load_assumptions()

In [ ]:
# Convert all MODIS HDFs (downloaded from EarthData into P.modis) to LC_Type1 GeoTIFFs
tif_folder = P.modis / "GeoTIFF"
drast.convert_modis_hdf_to_tif(P.modis, tif_folder)

### Use MODIS GeoTIFFs to predict land cover type for each GFA fire in USA (2003-2016) and compare them to given GFA land cover attribute. This predicts the land cover type by finding all MODIS landcover types inside each fire's boundary, then the most common one spatially is defined as that fire's predicted land cover type.

In [ ]:
y0, y1 = A["modis_years"]
gfa_global = dio.load_gfa(P.gfa, years=(y0, y1), bbox=dio.CONUS_BBOX, to_equal_area=False, verbose=False)
print(f"Number of GFA fires in USA ({y0}-{y1}): {len(gfa_global)}")

modis_by_year = drast.index_modis_by_year(tif_folder)
results_df = dcls.build_gfa_modis_validation(gfa_global, modis_by_year)

gfa_classified = results_df[results_df["area_km2"] >= A["min_fire"]].copy()

print(f"Fires > {A['min_fire']} km²: {len(gfa_classified)}")
gfa_classified.head()

In [ ]:
summary = rtab.classification_metrics(gfa_classified, ref_col="gfa_class", pred_col="modis_class")
summary

In [ ]:
gfa_classified = dcls.merge_classes(gfa_classified, ["modis_class", "gfa_class"])

dio.save_vector(gfa_classified, P.logged_output / "gfa_2003_2016_usa_classified_timevarying.shp")

This confusion matrix compares the given vs. predicted values in the categorical "land cover type" attribute. 

In [ ]:
confusion = pd.crosstab(gfa_classified["gfa_class"], gfa_classified["modis_class"])
confusion

In [ ]:
plc.plot_confusion_matrix(confusion)

In [ ]:
plc.plot_confusion_matrix(confusion, normalize=True)

In [ ]:
rtab.print_overall_metrics(confusion)

So, MODIS classifies fires by land cover type with relatively high accuracy. 

### Confirm that this prediction accuracy exists across the data's temporal range...

In [ ]:
periods = {
    "2003-2006": (2003, 2006),
    "2008-2011": (2008, 2011),
    "2013-2016": (2013, 2016)
}

metrics = plc.plot_confusion_by_period(gfa_classified, periods)
rtab.print_period_metrics(metrics)

Similar performance across temporal range!